In [2]:
import os
print(os.getcwd())
print(sorted(f for f in os.listdir(".") if f.endswith(".py")))

c:\Users\yrro\OneDrive\PGE2 COURSES\ML
['features.py', 'imbalance_experiment.py', 'imbivalance_experiment.py', 'pytest.py', 'test.data.py', 'test.features.py']


In [7]:
import credit_card_default.features as f
print([n for n in dir(f) if not n.startswith("_")])


['ALL_FEATURES', 'BASE_CAT', 'CREDIT_COLS', 'ColumnTransformer', 'OneHotEncoder', 'PAY', 'StandardScaler', 'TARGET', 'make_prep']


In [9]:
import sys
sys.path.append(r"C:\Users\yrro\OneDrive\PGE2 COURSES\ML\src")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, precision_recall_curve)
from credit_card_default.data import prepare_data
from credit_card_default.features import make_prep, add_features

df, X_train, X_test, y_train, y_test = prepare_data()
X_train_fe = add_features(X_train)                      # adds the 5 engineered columns
cv = StratifiedKFold(5, shuffle=True, random_state=42)

def base_models(class_weight=None):
    """Same settings as get_model in models.py; class_weight is what we vary here."""
    return {"logistic": LogisticRegression(class_weight=class_weight, max_iter=1000, random_state=42),
            "linear SVM": LinearSVC(class_weight=class_weight, max_iter=5000, random_state=42),
            "k-NN (k=50)": KNeighborsClassifier(n_neighbors=50)}

ImportError: cannot import name 'add_features' from 'credit_card_default.features' (C:\Users\yrro\OneDrive\PGE2 COURSES\ML\src\credit_card_default\features.py)

In [10]:
rows = {}
for name, clf in base_models().items():
    for pay_cat in (False, True):
        for eng in (False, True):
            pipe = Pipeline([("prep", make_prep(pay_cat, eng)), ("model", clf)])
            out = cross_validate(pipe, X_train_fe, y_train, cv=cv, scoring=["roc_auc", "average_precision"])
            rows[(name, "PAY one-hot" if pay_cat else "PAY numeric", "+ engineered" if eng else "base")] = {
                "AUC": out["test_roc_auc"].mean(), "AUC std": out["test_roc_auc"].std(),
                "AP": out["test_average_precision"].mean()}
pd.DataFrame(rows).T.round(4)

NameError: name 'base_models' is not defined

In [ ]:
def score_variant(pipe):
    model = pipe.steps[-1][1]
    method = "predict_proba" if hasattr(model, "predict_proba") else "decision_function"
    s = cross_val_predict(pipe, X_train_fe, y_train, cv=cv, method=method)
    if method == "predict_proba":
        s = s[:, 1]
    thr = 0.5 if method == "predict_proba" else 0.0
    pred = (s >= thr).astype(int)
    prec, rec, _ = precision_recall_curve(y_train, s)
    f1s = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    return {"AUC": roc_auc_score(y_train, s), "AP": average_precision_score(y_train, s),
            "recall@default": recall_score(y_train, pred), "precision@default": precision_score(y_train, pred, zero_division=0),
            "F1@default": f1_score(y_train, pred), "best F1": f1s.max(),
            "precision@recall>=.6": prec[rec >= 0.6].max(),
            "mean proba": s.mean() if method == "predict_proba" else np.nan}

rows = {}
for variant, cw in {"none": None, "class_weight": "balanced"}.items():
    for name, clf in base_models(cw).items():
        if variant == "class_weight" and name == "k-NN (k=50)":
            continue                                          # k-NN has no class_weight option
        rows[(name, variant)] = score_variant(Pipeline([("prep", make_prep(True, True)), ("model", clf)]))
pd.DataFrame(rows).T.sort_index().round(3)

c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6, 7, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6, 7, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded

AUC     AP  recall@0.5  precision@0.5  F1@0.5  \
logistic      SMOTE         0.772  0.538       0.610          0.472   0.532   
              class_weight  0.772  0.539       0.614          0.472   0.534   
              none          0.772  0.541       0.359          0.680   0.470   
random forest SMOTE         0.779  0.546       0.534          0.525   0.530   
              class_weight  0.785  0.558       0.590          0.501   0.542   
              none          0.784  0.558       0.371          0.672   0.478   

                            best F1  mean proba  
logistic      SMOTE           0.537       0.432  
              class_weight    0.539       0.434  
              none            0.539       0.221  
random forest SMOTE           0.535       0.351  
              class_weight    0.545       0.382  
              none            0.545       0.224

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

for name, clf in base_models().items():
    pipe = ImbPipeline([("prep", make_prep(True, True)), ("smote", SMOTE(random_state=42)), ("model", clf)])
    rows[(name, "SMOTE")] = score_variant(pipe)
pd.DataFrame(rows).T.sort_index().round(3)